# Using Psychological Factors to Predict League of Legends Player Rank

Investigating if using a classification model to predict the rank of a league of legends player based on psychological factors.

## 1. Data Loading & Initial Audit

Before cleaning or modeling, I loaded and explored the raw dataset. The questions I had were as follows:
1. What was the structural scale of the data?
1. Are there missing values located and how severe are they?
1. What do the distributions, scales, and extreme values look like across our features?

This data helped me figure out the optimal data classification split (rank 0 or rank 1), and how I needed to approach the pre-processing of this dataset

In [237]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/GamingStudy_data.csv', encoding='latin1')

In [238]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 13464 entries, 0 to 13463
Data columns (total 55 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       13464 non-null  int64  
 1   Zeitstempel      13464 non-null  float64
 2   GAD1             13464 non-null  int64  
 3   GAD2             13464 non-null  int64  
 4   GAD3             13464 non-null  int64  
 5   GAD4             13464 non-null  int64  
 6   GAD5             13464 non-null  int64  
 7   GAD6             13464 non-null  int64  
 8   GAD7             13464 non-null  int64  
 9   GADE             12815 non-null  str    
 10  SWL1             13464 non-null  int64  
 11  SWL2             13464 non-null  int64  
 12  SWL3             13464 non-null  int64  
 13  SWL4             13464 non-null  int64  
 14  SWL5             13464 non-null  int64  
 15  Game             13464 non-null  str    
 16  Platform         13464 non-null  str    
 17  Hours            13434 

In [239]:
print(df.isnull().sum()[df.isnull().sum() > 0])
print(df.isnull().sum().sum())

GADE                 649
Hours                 30
League              1852
highestleague      13464
streams              100
SPIN1                124
SPIN2                154
SPIN3                140
SPIN4                159
SPIN5                166
SPIN6                156
SPIN7                138
SPIN8                144
SPIN9                158
SPIN10               160
SPIN11               187
SPIN12               168
SPIN13               187
SPIN14               156
SPIN15               147
SPIN16               147
SPIN17               175
Narcissism            23
Work                  38
Degree              1577
Reference             15
accept               414
SPIN_T               650
Residence_ISO3       110
Birthplace_ISO3      121
dtype: int64
21709


In [240]:
print(df.describe().T[['count', 'mean', 'std', 'min', 'max']])

                 count          mean          std           min           max
Unnamed: 0     13464.0   7096.839201  4114.478220      1.000000  14250.000000
Zeitstempel    13464.0  42054.841222     0.272948  42052.004366  42058.363746
GAD1           13464.0      0.860963     0.926542      0.000000      3.000000
GAD2           13464.0      0.673351     0.915724      0.000000      3.000000
GAD3           13464.0      0.965761     0.982776      0.000000      3.000000
GAD4           13464.0      0.724079     0.921971      0.000000      3.000000
GAD5           13464.0      0.488042     0.837014      0.000000      3.000000
GAD6           13464.0      0.911022     0.931168      0.000000      3.000000
GAD7           13464.0      0.588755     0.894408      0.000000      3.000000
SWL1           13464.0      3.720440     1.736264      1.000000      7.000000
SWL2           13464.0      4.602050     1.696276      1.000000      7.000000
SWL3           13464.0      4.345440     1.809430      1.000000 

In [241]:
print(df[df['Game'] == 'League of Legends']['League'].value_counts().head(50))

League
Gold          845
Silver        592
Platinum      545
Diamond       421
gold          277
Unranked      235
Diamond 5     208
silver        195
Gold V        190
Gold 1        189
Gold 3        187
Gold 5        183
Silver 1      182
Silver 2      166
Gold 2        160
platinum      142
Silver 3      137
Bronze        127
diamond       117
Platinum 3    117
Gold 4        116
Silver 4      110
Platinum 2    109
Platinum 1    107
unranked      103
Platinum 5    100
Diamond V      97
Silver 5       94
Platinum 4     80
Platinum V     80
Diamond 3      77
Diamond 4      74
Gold IV        70
gold 5         64
Silver III     64
Diamond 1      61
Gold III       59
Bronze 1       55
Plat 5         53
Silver IV      50
gold 3         50
Silver V       48
Challenger     47
Silver I       47
Plat           46
Diamond 2      45
gold 2         45
Bronze 2       45
diamond 5      45
gold 1         44
Name: count, dtype: int64


## 2. Data Pre-Processing

Before putting anything into the training, the data must first be processed to only contain clean and relevant data.

This pre-processing code can be divided into **three** parts.

1. **Classification Target Mapping:** Convert messy free-text ranks into `League_Binary`
1. **Data Filtering:** Remove physically impossible datapoints and unnecessary metadata for predicting the target variable
1. **Data Verification:** Run another data audit after processing the dataframe to ensure nothing was missed

### Part 1: Map Free-Text to Binary Rank

This is the most important (and complex) step of the pre-processing procedure. The mapping function accounts for variations in spelling and casing, though edge cases exist (e.g, free-text entries citing past ranks like "Diamond 3 last season, placements currently"). The current logic matches the first recognizable tier string found. Will make this function more robust in future.

In [242]:
def map_league_to_binary(league_str):
    if pd.isna(league_str):
        return np.nan

    s = str(league_str).lower().strip()
    
    if any(tier in s for tier in ['platinum', 'plat', 'diamond', 'master', 'challenger', 'grandmaster']):
        return 1
    
    if any(tier in s for tier in ['bronze', 'silver', 'gold']):
        return 0
        
    return np.nan

In [243]:
df_lol = df[df['Game'] == 'League of Legends']

df_lol['League_Binary'] = df_lol['League'].apply(map_league_to_binary)

### Part 2: Data Filtering

Clean the dataset by removing physically impossible values. Remove duplicate rows and remove unecessary dataset to prevent noise in the model. Also removed rows with missing values. May in future use imputation to predict values, but for now I will just get the minimum working version

In [244]:
df_lol = df_lol[df_lol['Hours'] <= 112]

df_lol = df_lol.drop_duplicates().copy()

cols_to_drop = [
    'Unnamed: 0', 'Zeitstempel', 'highestleague', 'GADE', 'Game', 'Platform', 
    'streams', 'earnings', 'whyplay', 'League', 'Gender', 'Work', 'Degree', 
    'Birthplace', 'Residence', 'Reference', 'Playstyle', 'accept', 
    'Residence_ISO3', 'Birthplace_ISO3'
]

cols_to_drop_existing = [col for col in cols_to_drop if col in df_lol.columns]
df_clean = df_lol.drop(columns=cols_to_drop_existing).copy()

df_clean = df_clean.dropna().copy()

### Part 3: Post Processing Data Audit

For readability, I put this part at the very end. During development I ran the audit iteratively to catch mistakes made with processing early.

In [245]:
print(df_clean.info())

print(df_clean.describe().T[['count', 'mean', 'std', 'min', 'max']])

<class 'pandas.DataFrame'>
Index: 8918 entries, 11 to 13458
Data columns (total 36 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   GAD1           8918 non-null   int64  
 1   GAD2           8918 non-null   int64  
 2   GAD3           8918 non-null   int64  
 3   GAD4           8918 non-null   int64  
 4   GAD5           8918 non-null   int64  
 5   GAD6           8918 non-null   int64  
 6   GAD7           8918 non-null   int64  
 7   SWL1           8918 non-null   int64  
 8   SWL2           8918 non-null   int64  
 9   SWL3           8918 non-null   int64  
 10  SWL4           8918 non-null   int64  
 11  SWL5           8918 non-null   int64  
 12  Hours          8918 non-null   float64
 13  SPIN1          8918 non-null   float64
 14  SPIN2          8918 non-null   float64
 15  SPIN3          8918 non-null   float64
 16  SPIN4          8918 non-null   float64
 17  SPIN5          8918 non-null   float64
 18  SPIN6          8918 no

## 3. Data Split & Baseline

In [ ]:
x = df_clean.drop(columns=['League_Binary'])
y = df_clean['League_Binary']